# Intellicrack Hexcore Tutorial

A comprehensive, assertion-verified walkthrough of every public method in
`intellicrack_hexcore`, the Rust-backed (PyO3 / maturin) binary editing and
analysis engine.

**Verified specifications (from the current Rust source `src/intellicrack-hexcore/src/`):**

| Category | Count | Highlights |
|----------|------:|------------|
| Hash algorithms | 24 | MD5, SHA-1/2/3, BLAKE2, xxHash, SipHash, CRC, FNV, Adler32 |
| Transforms | 23 | XOR, ROT-N, AES-ECB (4 padding modes), Base64, zlib, bit/byte ops, masks |
| Built-in templates | 48 | 8 PE, 15 ELF, 15 Mach-O, 8 ZIP, 2 common |
| Encodings | 36 | UTF, ASCII (strict), EBCDIC, ISO-8859-x, Windows-125x, CJK, KOI8 |
| Inspector keys | 40+ | int8–64, float16–64, GUID, IPv4/6, timestamps |
| Patch formats | 6 | IPS, IPS32, COD, JSON, BPS, UPS |

`Vec<u8>` return types cross the PyO3 boundary as Python `bytes`; `Vec<u64>`
returns as `list[int]`. The tutorial assertions use both conventions where
appropriate.

Every code cell includes assertions that **must** pass against the current
build. A failure indicates either a drift between these assertions and the
Rust implementation or a genuine bug in the hexcore implementation.

## 0. Setup & Import

In [ ]:
import intellicrack_hexcore
from intellicrack_hexcore import HexDocument, Bookmark, diff_bytes, diff_files

HELLO_WORLD = b"Hello World!"

exports = [x for x in dir(intellicrack_hexcore) if not x.startswith("_")]
print(f"Module: {intellicrack_hexcore.__name__}")
print(f"Exports: {exports}")
assert "HexDocument" in exports
assert "Bookmark" in exports
assert "diff_bytes" in exports
assert "diff_files" in exports
print("Setup OK")

## 1. Creating Documents

Three constructors:
- `HexDocument()` — empty document (0 bytes)
- `HexDocument.open_bytes(data)` — from a `bytes` object (static method)
- `HexDocument.open(path)` — memory-maps a file on disk (static method)

In [ ]:
doc_empty = HexDocument()
assert doc_empty.length() == 0
assert doc_empty.file_path() is None

sample = b"Hello, Hexcore!"
doc = HexDocument.open_bytes(sample)
assert doc.length() == len(sample)
assert doc.file_path() is None

print(f"Empty document: {doc_empty.length()} bytes")
print(f"From bytes:     {doc.length()} bytes")
print("Creating documents OK")

## 2. Reading & Writing

- `read(offset, length)` → `bytes` (exact slice; truncated to document end)
- `read_byte(offset)` → `int`
- `write_bytes(offset, data)` — overwrites in-place; length is clamped to
  fit within the document. Errors if `offset >= length()`. Recorded for undo.
- `length()` → `int`

In [ ]:
doc = HexDocument.open_bytes(b"ABCDEFGHIJ")

data = doc.read(0, 5)
assert isinstance(data, bytes)
assert data == b"ABCDE"

byte_val = doc.read_byte(0)
assert byte_val == 0x41

assert doc.length() == 10

doc.write_bytes(0, b"XY")
assert doc.read(0, 2) == b"XY"
assert doc.read(2, 3) == b"CDE"
assert doc.length() == 10

print(f"Read 5 bytes: {data}")
print(f"Single byte at 0: 0x{byte_val:02X}")
print(f"After overwrite: {doc.read(0, 10)}")
print("Reading & writing OK")

## 3. Insert & Delete

- `insert_bytes(offset, data)` — inserts bytes, shifting subsequent data
- `delete_bytes(offset, length)` — removes bytes, shifting subsequent data

In [ ]:
doc = HexDocument.open_bytes(b"ABCDEF")
assert doc.length() == 6

doc.insert_bytes(3, b"XYZ")
assert doc.length() == 9
assert doc.read(0, 9) == b"ABCXYZDEF"

doc.delete_bytes(3, 3)
assert doc.length() == 6
assert doc.read(0, 6) == b"ABCDEF"

print("Insert & delete OK")

## 4. Undo / Redo

- `undo()` / `redo()` → `bool` (whether an operation was undone/redone)
- `can_undo()` / `can_redo()` → `bool`
- `is_modified()` → `bool`

Undo records cover overwrites, inserts, deletes, bit operations, block
operations, and `repair_pe_checksum`. `import_patches_ips` records each
applied record. `import_patches_bps` / `import_patches_ups` replace the
entire document and reset the undo stack.

In [ ]:
doc = HexDocument.open_bytes(b"ORIGINAL")
assert not doc.is_modified()
assert not doc.can_undo()
assert not doc.can_redo()

doc.write_bytes(0, b"MODIFIED")
assert doc.is_modified()
assert doc.can_undo()
assert doc.read(0, 8) == b"MODIFIED"

assert doc.undo()
assert doc.read(0, 8) == b"ORIGINAL"
assert doc.can_redo()

assert doc.redo()
assert doc.read(0, 8) == b"MODIFIED"

print("Undo / redo OK")

## 5. Save & File Path

- `save(path)` / `save_as(path)` — atomic temp-file + rename; remaps the
  document at the new path and clears the modified flag
- `file_path()` → `str | None`

After a successful `save`/`save_as`, `file_path()` returns the saved path.

In [ ]:
import os
import tempfile

doc = HexDocument.open_bytes(b"Save test data")
assert doc.file_path() is None

with tempfile.NamedTemporaryFile(suffix=".bin", delete=False) as f:
    tmp_path = f.name

copy_path = tmp_path + ".copy"
try:
    doc.save(tmp_path)
    assert doc.file_path() is not None
    assert os.path.samefile(doc.file_path(), tmp_path)
    assert not doc.is_modified()

    doc2 = HexDocument.open(tmp_path)
    assert doc2.length() == doc.length()
    assert doc2.read(0, doc2.length()) == b"Save test data"
    assert doc2.file_path() is not None
    print(f"Saved to: {doc2.file_path()}")

    doc2.save_as(copy_path)
    assert os.path.exists(copy_path)
    assert os.path.samefile(doc2.file_path(), copy_path)
finally:
    if os.path.exists(copy_path):
        os.unlink(copy_path)
    if os.path.exists(tmp_path):
        os.unlink(tmp_path)

print("Save & file path OK")

## 6. Byte Search

- `search_bytes(pattern, max_results)` → `list[tuple[offset, length]]`
- `search_hex(pattern, max_results)` → same — hex string with `?` wildcards
  per nibble; whitespace between bytes is ignored

In [ ]:
pe_header = b"MZ" + b"\x00" * 58 + b"\x80\x00\x00\x00" + b"PE\x00\x00" + b"\x00" * 28 + b"MZ_end"
doc = HexDocument.open_bytes(pe_header)

results = doc.search_bytes(b"MZ", 10)
assert len(results) >= 1
assert results[0] == (0, 2)
print(f"search_bytes(b'MZ'): {results}")

hex_results = doc.search_hex("4D 5A", 10)
assert len(hex_results) >= 1
assert hex_results[0] == (0, 2)
print(f"search_hex('4D 5A'): {hex_results}")

wildcard_results = doc.search_hex("4D ?A", 10)
assert len(wildcard_results) >= 1
print(f"search_hex('4D ?A') wildcard: {wildcard_results}")

pe_results = doc.search_bytes(b"PE\x00\x00", 10)
assert len(pe_results) >= 1
print(f"search_bytes(PE sig): {pe_results}")

print("Byte search OK")

## 7. Text Search

- `search_text(text, encoding, case_sensitive, max_results)` →
  `list[tuple[offset, length]]`
- `search_text_encoded(text, encoding, case_sensitive, max_results)` → same,
  encoding-aware (UTF-16, Shift_JIS, EBCDIC, etc.)

Case-insensitive matching handles mixed-case input patterns correctly.

In [ ]:
text_data = b"Hello World! hello again. HELLO FINAL."
doc = HexDocument.open_bytes(text_data)

case_sensitive = doc.search_text("Hello", "utf-8", True, 10)
assert len(case_sensitive) == 1
assert case_sensitive[0] == (0, 5)
print(f"Case-sensitive 'Hello': {case_sensitive}")

case_insensitive = doc.search_text("hello", "utf-8", False, 10)
assert len(case_insensitive) == 3
print(f"Case-insensitive 'hello': {case_insensitive}")

mixed_case = doc.search_text("HeLLo", "utf-8", False, 10)
assert len(mixed_case) == 3
print(f"Case-insensitive mixed-case 'HeLLo': {mixed_case}")

utf16_payload = "Hello".encode("utf-16-le") + b"\x00" * 10 + "Hello".encode("utf-16-le")
doc2 = HexDocument.open_bytes(utf16_payload)
encoded_results = doc2.search_text_encoded("Hello", "utf-16le", True, 10)
assert len(encoded_results) == 2
print(f"search_text_encoded UTF-16LE: {encoded_results}")

print("Text search OK")

## 8. Regex Search

- `search_regex(pattern, max_results)` → `list[tuple[offset, length]]`

Patterns use Rust `regex` crate syntax (ASCII-byte matching over raw data).

In [ ]:
data = b"Error: code=404, Error: code=500, Info: code=200"
doc = HexDocument.open_bytes(data)

results = doc.search_regex(r"code=\d+", 10)
assert len(results) == 3
print(f"Regex 'code=\\d+': {results}")
for offset, length in results:
    print(f"  offset={offset}: {doc.read(offset, length).decode('ascii')}")

email_data = b"Contact alice@example.com or bob@test.org for info"
doc2 = HexDocument.open_bytes(email_data)
email_results = doc2.search_regex(r"[a-zA-Z]+@[a-zA-Z]+\.[a-zA-Z]+", 10)
assert len(email_results) == 2
print(f"Email regex: {email_results}")

print("Regex search OK")

## 9. Numeric Search

- `search_numeric(value, size, signed, big_endian, alignment, max_results)`
- `search_numeric_float(value, size, big_endian, tolerance, alignment, max_results)`
- `search_numeric_range(value_range, size, signed, big_endian, alignment, max_results)`

All return `list[tuple[offset, length]]`. `size` is the target width in bytes
(1, 2, 4, 8 for integers; 4 or 8 for floats). `alignment` restricts hits to
offsets divisible by the given value.

In [ ]:
import struct

values = [42, 1000, 42, 65535, 42]
data = struct.pack("<5i", *values)
doc = HexDocument.open_bytes(data)

int_results = doc.search_numeric(42, 4, True, False, 4, 10)
assert len(int_results) == 3
print(f"search_numeric(42, i32 LE): {int_results}")

float_values = [3.14, 2.71, 3.14, 1.41]
float_data = struct.pack("<4f", *float_values)
doc2 = HexDocument.open_bytes(float_data)
float_results = doc2.search_numeric_float(3.14, 4, False, 0.01, 4, 10)
assert len(float_results) == 2
print(f"search_numeric_float(3.14, f32 LE): {float_results}")

range_data = struct.pack("<4i", 10, 100, 200, 300)
doc3 = HexDocument.open_bytes(range_data)
range_results = doc3.search_numeric_range((50, 250), 4, True, False, 4, 10)
assert len(range_results) >= 2
print(f"search_numeric_range(50..250, i32 LE): {range_results}")

print("Numeric search OK")

## 10. Find & Replace

- `replace_bytes(pattern, replacement)` → `int` (count of replacements)

The entire document is rebuilt when replacements occur and the change is
recorded as a single overwrite for undo purposes.

In [ ]:
doc = HexDocument.open_bytes(b"foo bar foo baz foo")
count = doc.replace_bytes(b"foo", b"FOO")
assert count == 3
assert doc.read(0, doc.length()) == b"FOO bar FOO baz FOO"
print(f"Replaced {count} occurrences: {doc.read(0, doc.length())}")

count2 = doc.replace_bytes(b"notfound", b"X")
assert count2 == 0
print(f"No-match replace count: {count2}")

print("Find & replace OK")

## 11. Data Inspector

`inspect_at(offset)` returns a `dict[str, str]` with up to 40+ interpretations
depending on the number of available bytes:

| Bytes | Keys |
|------:|------|
| 1 | int8, uint8, ascii_char, utf8_char, uleb128, sleb128 |
| 2 | int16_le/be, uint16_le/be, float16_le/be, rgb565, dos_time, dos_date |
| 3 | int24_le/be, uint24_le/be |
| 4 | int32_le/be, uint32_le/be, float32_le/be, rgba8, ipv4, unix_timestamp |
| 6 | int48_le/be, uint48_le/be |
| 8 | int64_le/be, uint64_le/be, float64_le/be, filetime |
| 16 | guid, ipv6 |
| 2+ | wide_string (UTF-16LE, up to 32 code units) |

Timestamp, float, DOS date/time, and wide_string entries are only emitted
when the raw bytes decode to a plausible value.

In [ ]:
import struct

doc = HexDocument.open_bytes(bytes([0x41]) + b"\x00" * 15)
insp = doc.inspect_at(0)
assert isinstance(insp, dict)
assert insp["uint8"] == "65"
assert insp["int8"] == "65"
assert insp["ascii_char"] == "A"
assert insp["uint16_le"] == "65"
print(f"Keys at offset 0: {sorted(insp.keys())}")
print(f"  uint8={insp['uint8']}, ascii_char={insp['ascii_char']}")

ip_data = bytes([192, 168, 1, 1]) + b"\x00" * 12
doc2 = HexDocument.open_bytes(ip_data)
insp2 = doc2.inspect_at(0)
assert insp2["ipv4"] == "192.168.1.1"
print(f"  ipv4={insp2['ipv4']}")

guid_data = bytes([0x01, 0x02, 0x03, 0x04, 0x05, 0x06, 0x07, 0x08,
                   0x09, 0x0A, 0x0B, 0x0C, 0x0D, 0x0E, 0x0F, 0x10])
doc3 = HexDocument.open_bytes(guid_data)
insp3 = doc3.inspect_at(0)
assert insp3["guid"] == "04030201-0605-0807-090a-0b0c0d0e0f10"
print(f"  guid={insp3['guid']}")

ipv6_data = bytes([0x20, 0x01, 0x0d, 0xb8, 0x00, 0x00, 0x00, 0x00,
                   0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x01])
doc4 = HexDocument.open_bytes(ipv6_data)
insp4 = doc4.inspect_at(0)
assert insp4["ipv6"] == "2001:db8:0:0:0:0:0:1"
print(f"  ipv6={insp4['ipv6']}")

ts_val = 1_704_067_200
ts_data = struct.pack("<I", ts_val) + b"\x00" * 12
doc5 = HexDocument.open_bytes(ts_data)
insp5 = doc5.inspect_at(0)
assert "unix_timestamp" in insp5
assert insp5["unix_timestamp"].startswith("2024-01-01")
print(f"  unix_timestamp={insp5['unix_timestamp']}")

print("Data inspector OK")

## 12. Hashing

24 hash algorithms, each accepting one or more case-insensitive name
variants (hyphenated and underscored aliases supported). All verified
against known test vectors from the Rust test suite.

| Category | Algorithms |
|----------|-----------|
| Cryptographic | md5, sha1, sha224, sha256, sha384, sha512, sha3-256, sha3-512, blake2b, blake2s |
| Non-crypto | xxhash32, xxhash64, xxh3, siphash64, siphash128 |
| Checksums | adler32, crc8, crc16, crc32, crc64 |
| FNV | fnv1-32, fnv1-64, fnv1a-32, fnv1a-64 |

In [ ]:
doc_abc = HexDocument.open_bytes(b"abc")
doc_empty = HexDocument.open_bytes(b"")
doc_digits = HexDocument.open_bytes(b"123456789")

assert doc_abc.compute_hash("md5") == "900150983cd24fb0d6963f7d28e17f72"
assert doc_abc.compute_hash("sha1") == "a9993e364706816aba3e25717850c26c9cd0d89d"
assert doc_abc.compute_hash("sha224") == "23097d223405d8228642a477bda255b32aadbce4bda0b3f7e36c9da7"
assert doc_abc.compute_hash("sha256") == "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"
assert doc_abc.compute_hash("sha384").startswith("cb00753f45a35e8b")
assert doc_abc.compute_hash("sha512").startswith("ddaf35a193617aba")
print("Cryptographic hashes: MD5, SHA-1, SHA-224, SHA-256, SHA-384, SHA-512 OK")

assert doc_abc.compute_hash("sha3-256") == doc_abc.compute_hash("sha3_256")
assert doc_abc.compute_hash("sha3-512") == doc_abc.compute_hash("sha3_512")
assert doc_empty.compute_hash("sha3-256") == (
    "a7ffc6f8bf1ed76651c14756a061d662f580ff4de43b49fa82d80a4b80f8434a"
)
assert len(doc_abc.compute_hash("blake2b")) == 64
assert len(doc_abc.compute_hash("blake2s")) == 64
assert doc_abc.compute_hash("blake2b") == doc_abc.compute_hash("blake2b-256")
assert doc_abc.compute_hash("blake2b") == doc_abc.compute_hash("blake2b256")
print("SHA-3, BLAKE2 OK")

assert doc_empty.compute_hash("xxhash32") == "02cc5d05"
assert doc_empty.compute_hash("xxhash32") == doc_empty.compute_hash("xxh32")
assert doc_empty.compute_hash("xxhash64") == "ef46db3751d8e999"
assert doc_empty.compute_hash("xxhash64") == doc_empty.compute_hash("xxh64")
assert len(doc_abc.compute_hash("xxh3")) == 16
assert len(doc_abc.compute_hash("siphash64")) == 16
assert doc_abc.compute_hash("siphash64") == doc_abc.compute_hash("siphash")
assert len(doc_abc.compute_hash("siphash128")) == 32
print("xxHash, SipHash OK")

assert doc_empty.compute_hash("adler32") == "00000001"
assert len(doc_digits.compute_hash("crc8")) == 2
assert len(doc_digits.compute_hash("crc16")) == 4
assert doc_digits.compute_hash("crc32") == "cbf43926"
assert doc_empty.compute_hash("crc32") == "00000000"
assert len(doc_digits.compute_hash("crc64")) == 16
assert doc_digits.compute_hash("crc64") == doc_digits.compute_hash("crc64-ecma")
print("Adler32, CRC-8/16/32/64 OK")

assert doc_empty.compute_hash("fnv1-32") == "811c9dc5"
assert doc_empty.compute_hash("fnv1-64") == "cbf29ce484222325"
assert doc_empty.compute_hash("fnv1a-32") == "811c9dc5"
assert doc_empty.compute_hash("fnv1a-64") == "cbf29ce484222325"
print("FNV-1/1a OK")

assert doc_abc.compute_hash("SHA256") == doc_abc.compute_hash("sha256")
assert doc_abc.compute_hash("Sha3-256") == doc_abc.compute_hash("sha3-256")
print("Case-insensitive algorithm names OK")
print("\nAll 24 hash algorithms verified")

In [ ]:
doc = HexDocument.open_bytes(b"Hello World")
full_hash = HexDocument.open_bytes(b"World").compute_hash("sha256")
range_hash = doc.compute_hash_range(6, 11, "sha256")
assert full_hash == range_hash
print(f"compute_hash_range [6:11] matches full-data hash: {range_hash[:32]}...")

crc_doc = HexDocument.open_bytes(b"123456789")
custom_crc = crc_doc.compute_hash_custom_crc(
    (0, 9),
    0x04C11DB7,
    0xFFFFFFFF,
    32,
    (True, True),
    0xFFFFFFFF,
)
assert custom_crc == "cbf43926"
print(f"compute_hash_custom_crc (CRC-32 standard params): {custom_crc}")

print("Hash range & custom CRC OK")

## 13. Entropy Analysis

- `entropy()` → Shannon entropy of entire document (0.0 = uniform, 8.0 = max)
- `entropy_map(block_size)` → `list[float]` per-block entropy values
- `byte_distribution_full()` → `list[int]` with 256 entries (u64 counts)
- `byte_type_distribution()` → `tuple[int, int, int, int]` —
  `(null, printable, control, high_byte)`
- `digram_matrix()` → `list[int]` with 65536 entries (256×256 bigram counts)
- `content_classification(block_size)` → `bytes` — per-block class codes
  `0=null, 1=text, 2=mixed, 3=high-entropy, 4=moderate-high`

In [ ]:
null_block = b"\x00" * 1024
text_block = b"The quick brown fox jumps over the lazy dog. " * 23
prng_block = bytearray(1024)
state = 12345
for i in range(1024):
    state = (state * 1103515245 + 12345) & 0xFFFFFFFF
    prng_block[i] = (state >> 16) & 0xFF

doc_null = HexDocument.open_bytes(null_block)
ent_null = doc_null.entropy()
assert ent_null < 0.01
print(f"Null block entropy:  {ent_null:.4f}")

doc_text = HexDocument.open_bytes(text_block)
ent_text = doc_text.entropy()
assert 3.0 < ent_text < 6.0
print(f"Text block entropy:  {ent_text:.4f}")

doc_prng = HexDocument.open_bytes(bytes(prng_block))
ent_prng = doc_prng.entropy()
assert ent_prng > 6.5
print(f"PRNG block entropy:  {ent_prng:.4f}")

combined = null_block + text_block[:1024] + bytes(prng_block)
doc_combined = HexDocument.open_bytes(combined)

emap = doc_combined.entropy_map(1024)
assert len(emap) == 3
assert emap[0] < 0.01
assert emap[1] > 3.0
assert emap[2] > 6.5
print(f"Entropy map (3 blocks): [{emap[0]:.2f}, {emap[1]:.2f}, {emap[2]:.2f}]")

dist = doc_prng.byte_distribution_full()
assert len(dist) == 256
assert sum(dist) == 1024
print(f"Byte distribution: {len(dist)} entries, total={sum(dist)}")

null_count, printable_count, control_count, high_count = doc_text.byte_type_distribution()
assert printable_count > null_count
print(f"Byte types (text): null={null_count}, printable={printable_count}, "
      f"control={control_count}, high={high_count}")

digram = doc_text.digram_matrix()
assert len(digram) == 65536
print(f"Digram matrix: {len(digram)} entries (256x256)")

classes = doc_combined.content_classification(1024)
assert isinstance(classes, bytes)
assert len(classes) == 3
assert classes[0] == 0
assert classes[1] == 1
assert classes[2] in (3, 4)
CLASS_NAMES = {0: "null", 1: "text", 2: "mixed", 3: "high-entropy", 4: "moderate-high"}
print(f"Content classification: {[CLASS_NAMES.get(c, '?') for c in classes]}")

print("Entropy analysis OK")

## 14. Transforms

23 transforms across 7 categories, applied via:

```python
doc.transform_data(name, offset, length, params: dict[str, bytes]) -> bytes
HexDocument.list_transforms() -> list[tuple[name, category, description]]
```

`list_transforms` is a static method. `transform_data` returns the transformed
bytes; it does **not** mutate the document.

In [ ]:
transforms = HexDocument.list_transforms()
assert len(transforms) == 23
print(f"Total transforms: {len(transforms)}")
categories: dict[str, list[str]] = {}
for name, category, desc in transforms:
    categories.setdefault(category, []).append(name)
assert set(categories.keys()) == {
    "xor", "cipher", "encoding", "compression", "bitops", "byteops", "mask",
}
for cat, names in sorted(categories.items()):
    print(f"  [{cat}] {', '.join(names)}")

### 14a. XOR Transforms

In [ ]:
doc = HexDocument.open_bytes(HELLO_WORLD)

encrypted = doc.transform_data("xor_single", 0, doc.length(), {"key": b"\x42"})
assert isinstance(encrypted, bytes)
doc2 = HexDocument.open_bytes(encrypted)
decrypted = doc2.transform_data("xor_single", 0, doc2.length(), {"key": b"\x42"})
assert decrypted == HELLO_WORLD
print("xor_single roundtrip OK")

enc = doc.transform_data("xor_repeating", 0, doc.length(), {"key": b"KEY"})
doc3 = HexDocument.open_bytes(enc)
dec = doc3.transform_data("xor_repeating", 0, doc3.length(), {"key": b"KEY"})
assert dec == HELLO_WORLD
print("xor_repeating roundtrip OK")

doc4 = HexDocument.open_bytes(b"\x00\x00\x00\x00")
rolled = doc4.transform_data("xor_rolling", 0, 4, {"key": b"\x10", "increment": b"\x01"})
assert rolled == bytes([0x10, 0x11, 0x12, 0x13])
print(f"xor_rolling: {list(rolled)}")

doc5 = HexDocument.open_bytes(b"\x00\x00\x00")
rolled_default = doc5.transform_data("xor_rolling", 0, 3, {"key": b"\x10"})
assert rolled_default == bytes([0x10, 0x11, 0x12])
print("xor_rolling default increment=1 OK")

### 14b. Cipher Transforms

`rot_n` applies to ASCII letters only; `shift` mod 26. `aes_ecb_encrypt` and
`aes_ecb_decrypt` accept 16/24/32-byte keys and a `padding` parameter with
four modes:

- `b"none"` — input must already be a multiple of 16 bytes
- `b"pkcs7"` — **default**; 1–16 bytes appended, strictly validated on decrypt
- `b"zero"` — trailing zeros on encrypt; plaintext returned as-is on decrypt
- `b"iso10126"` — filler bytes + length byte (implementation uses deterministic zero filler), strictly validated on decrypt

PKCS7 always adds padding (a full block when the input is already aligned),
so a 16-byte plaintext produces a 32-byte ciphertext under the default mode.

In [ ]:
doc = HexDocument.open_bytes(b"Hello World!!!!!")
rotated = doc.transform_data("rot_n", 0, doc.length(), {"shift": b"\x0d"})
doc2 = HexDocument.open_bytes(rotated)
unrotated = doc2.transform_data("rot_n", 0, doc2.length(), {"shift": b"\x0d"})
assert unrotated == b"Hello World!!!!!"
print(f"ROT13: {rotated.decode('ascii')} -> roundtrip OK")

key_128 = b"\x00" * 16
aes_data = b"A" * 16

doc3 = HexDocument.open_bytes(aes_data)
encrypted_none = doc3.transform_data(
    "aes_ecb_encrypt", 0, 16,
    {"key": key_128, "padding": b"none"},
)
assert len(encrypted_none) == 16
assert encrypted_none != aes_data
doc4 = HexDocument.open_bytes(encrypted_none)
decrypted_none = doc4.transform_data(
    "aes_ecb_decrypt", 0, 16,
    {"key": key_128, "padding": b"none"},
)
assert decrypted_none == aes_data
print("AES-128 ECB (padding=none) roundtrip OK")

pkcs_plaintext = b"Short data"
doc5 = HexDocument.open_bytes(pkcs_plaintext)
encrypted_pkcs = doc5.transform_data(
    "aes_ecb_encrypt", 0, len(pkcs_plaintext),
    {"key": key_128},
)
assert len(encrypted_pkcs) == 16
doc6 = HexDocument.open_bytes(encrypted_pkcs)
decrypted_pkcs = doc6.transform_data(
    "aes_ecb_decrypt", 0, len(encrypted_pkcs),
    {"key": key_128},
)
assert decrypted_pkcs == pkcs_plaintext
print("AES-128 ECB (default PKCS7) roundtrip OK")

key_256 = b"\x01" * 32
doc7 = HexDocument.open_bytes(aes_data)
enc256 = doc7.transform_data(
    "aes_ecb_encrypt", 0, 16,
    {"key": key_256, "padding": b"none"},
)
doc8 = HexDocument.open_bytes(enc256)
dec256 = doc8.transform_data(
    "aes_ecb_decrypt", 0, 16,
    {"key": key_256, "padding": b"none"},
)
assert dec256 == aes_data
print("AES-256 ECB (padding=none) roundtrip OK")

### 14c. Encoding & Compression Transforms

In [ ]:
doc = HexDocument.open_bytes(HELLO_WORLD)

encoded = doc.transform_data("base64_encode", 0, doc.length(), {})
assert encoded == b"SGVsbG8gV29ybGQh"
doc2 = HexDocument.open_bytes(encoded)
decoded = doc2.transform_data("base64_decode", 0, doc2.length(), {})
assert decoded == HELLO_WORLD
print(f"Base64: {encoded.decode('ascii')} -> roundtrip OK")

big_data = b"AAAA" * 256
doc3 = HexDocument.open_bytes(big_data)
compressed = doc3.transform_data("zlib_deflate", 0, doc3.length(), {})
assert len(compressed) < len(big_data)
doc4 = HexDocument.open_bytes(compressed)
decompressed = doc4.transform_data("zlib_inflate", 0, doc4.length(), {})
assert decompressed == big_data
ratio = 100 * len(compressed) / len(big_data)
print(f"Zlib: {len(big_data)} -> {len(compressed)} bytes ({ratio:.1f}%) -> roundtrip OK")

### 14d. Bit Operation Transforms

`bit_shift_left` and `bit_shift_right` require `count` in `0..=7`; counts
greater than 7 are rejected with an `InvalidParameter` error. `bit_rotate_left`
and `bit_rotate_right` take `count` modulo 8.

In [ ]:
doc = HexDocument.open_bytes(bytes([0b00000001]))
shifted = doc.transform_data("bit_shift_left", 0, 1, {"count": b"\x02"})
assert list(shifted) == [0b00000100]
print(f"bit_shift_left(0x01, 2): 0b{shifted[0]:08b}")

doc2 = HexDocument.open_bytes(bytes([0b10000000]))
shifted_r = doc2.transform_data("bit_shift_right", 0, 1, {"count": b"\x03"})
assert list(shifted_r) == [0b00010000]
print(f"bit_shift_right(0x80, 3): 0b{shifted_r[0]:08b}")

doc3 = HexDocument.open_bytes(bytes([0b10000001]))
rotated = doc3.transform_data("bit_rotate_left", 0, 1, {"count": b"\x01"})
assert list(rotated) == [0b00000011]
print(f"bit_rotate_left(0x81, 1): 0b{rotated[0]:08b}")

doc4 = HexDocument.open_bytes(bytes([0b10000001]))
rotated_r = doc4.transform_data("bit_rotate_right", 0, 1, {"count": b"\x01"})
assert list(rotated_r) == [0b11000000]
print(f"bit_rotate_right(0x81, 1): 0b{rotated_r[0]:08b}")

doc5 = HexDocument.open_bytes(b"Test")
inverted = doc5.transform_data("bit_invert", 0, 4, {})
doc6 = HexDocument.open_bytes(inverted)
restored = doc6.transform_data("bit_invert", 0, 4, {})
assert restored == b"Test"
print("bit_invert roundtrip OK")

### 14e. Byte Operation Transforms

In [ ]:
doc = HexDocument.open_bytes(b"ABCDE")
reversed_data = doc.transform_data("byte_reverse", 0, 5, {})
assert reversed_data == b"EDCBA"
print(f"byte_reverse: {reversed_data}")

doc2 = HexDocument.open_bytes(bytes([0x01, 0x02, 0x03, 0x04]))
swapped = doc2.transform_data("byte_swap_16", 0, 4, {})
assert list(swapped) == [0x02, 0x01, 0x04, 0x03]
print(f"byte_swap_16: {list(swapped)}")

doc3 = HexDocument.open_bytes(bytes([0x01, 0x02, 0x03, 0x04]))
swapped32 = doc3.transform_data("byte_swap_32", 0, 4, {})
assert list(swapped32) == [0x04, 0x03, 0x02, 0x01]
print(f"byte_swap_32: {list(swapped32)}")

doc4 = HexDocument.open_bytes(bytes(range(8)))
swapped64 = doc4.transform_data("byte_swap_64", 0, 8, {})
assert list(swapped64) == [7, 6, 5, 4, 3, 2, 1, 0]
print(f"byte_swap_64: {list(swapped64)}")

doc5 = HexDocument.open_bytes(bytes([0x41, 0x00, 0x42, 0x00, 0x43]))
cleaned = doc5.transform_data("remove_nulls", 0, 5, {})
assert cleaned == b"ABC"
print(f"remove_nulls: {cleaned}")

### 14f. Mask Transforms

In [ ]:
doc = HexDocument.open_bytes(bytes([0xFF, 0xFF, 0xFF, 0xFF]))
anded = doc.transform_data("mask_and", 0, 4, {"pattern": b"\x0F"})
assert list(anded) == [0x0F, 0x0F, 0x0F, 0x0F]
print(f"mask_and(0xFF, 0x0F): {[f'0x{b:02X}' for b in anded]}")

doc2 = HexDocument.open_bytes(bytes([0x00, 0x00, 0x00, 0x00]))
ored = doc2.transform_data("mask_or", 0, 4, {"pattern": b"\xF0"})
assert list(ored) == [0xF0, 0xF0, 0xF0, 0xF0]
print(f"mask_or(0x00, 0xF0): {[f'0x{b:02X}' for b in ored]}")

doc3 = HexDocument.open_bytes(b"AAAA")
xored = doc3.transform_data("mask_xor", 0, 4, {"pattern": b"\x20"})
assert xored == b"aaaa"
print(f"mask_xor('AAAA', 0x20): {xored}")

print("All 23 transforms verified")

## 15. Encodings

36 encodings via:
- `HexDocument.list_encodings()` → `list[tuple[name, description]]` (static)
- `doc.decode_text(offset, length, encoding)` → `str`
- `HexDocument.encode_text_to_bytes(text, encoding)` → `bytes` (static)
- `doc.search_text_encoded(text, encoding, case_sensitive, max_results)`

The `ascii` encoder is strict: input characters outside 0x00–0x7F are
rejected rather than silently replaced.

In [ ]:
encodings = HexDocument.list_encodings()
assert len(encodings) == 36
print(f"Supported encodings: {len(encodings)}")
for name, desc in encodings:
    print(f"  {name}: {desc}")

In [ ]:
test_text = "Hello, World!"

encoded = HexDocument.encode_text_to_bytes(test_text, "utf-8")
assert isinstance(encoded, bytes)
doc = HexDocument.open_bytes(encoded)
decoded = doc.decode_text(0, len(encoded), "utf-8")
assert decoded == test_text
print(f"UTF-8 roundtrip OK ({len(encoded)} bytes)")

encoded_u16 = HexDocument.encode_text_to_bytes(test_text, "utf-16le")
doc2 = HexDocument.open_bytes(encoded_u16)
decoded_u16 = doc2.decode_text(0, len(encoded_u16), "utf-16le")
assert decoded_u16 == test_text
print(f"UTF-16LE roundtrip OK ({len(encoded_u16)} bytes)")

encoded_ebc = HexDocument.encode_text_to_bytes("Hello World 0123", "ebcdic")
doc3 = HexDocument.open_bytes(encoded_ebc)
decoded_ebc = doc3.decode_text(0, len(encoded_ebc), "ebcdic")
assert decoded_ebc == "Hello World 0123"
print(f"EBCDIC roundtrip OK ({len(encoded_ebc)} bytes)")

encoded_1252 = HexDocument.encode_text_to_bytes("caf\u00e9", "windows-1252")
doc4 = HexDocument.open_bytes(encoded_1252)
decoded_1252 = doc4.decode_text(0, len(encoded_1252), "windows-1252")
assert decoded_1252 == "caf\u00e9"
print("Windows-1252 roundtrip OK")

encoded_sjis = HexDocument.encode_text_to_bytes("Hello", "shift_jis")
doc5 = HexDocument.open_bytes(encoded_sjis)
decoded_sjis = doc5.decode_text(0, len(encoded_sjis), "shift_jis")
assert decoded_sjis == "Hello"
print("Shift_JIS roundtrip OK")

encoded_ascii = HexDocument.encode_text_to_bytes("plain ASCII", "ascii")
assert encoded_ascii == b"plain ASCII"
try:
    HexDocument.encode_text_to_bytes("caf\u00e9", "ascii")
except ValueError as exc:
    print(f"Strict ASCII rejects non-ASCII input: {type(exc).__name__}")
else:
    raise AssertionError("strict ASCII encoder must reject non-ASCII input")

utf16_payload = HexDocument.encode_text_to_bytes("Hello World Hello", "utf-16le")
doc6 = HexDocument.open_bytes(utf16_payload)
results = doc6.search_text_encoded("Hello", "utf-16le", True, 10)
assert len(results) == 2
print(f"search_text_encoded UTF-16LE: {results}")

print("Encodings OK")

## 16. Binary Templates

48 built-in templates covering PE, ELF (LE and BE), Mach-O (32/64-bit, fat,
big-endian variants, and load commands), ZIP/ZIP64, and common composite
types:

- **PE (8):** `IMAGE_DOS_HEADER`, `IMAGE_FILE_HEADER`,
  `IMAGE_OPTIONAL_HEADER32`, `IMAGE_OPTIONAL_HEADER64`,
  `IMAGE_SECTION_HEADER`, `IMAGE_DATA_DIRECTORY`,
  `IMAGE_IMPORT_DESCRIPTOR`, `IMAGE_EXPORT_DIRECTORY`
- **ELF (15):** `Elf32_Ehdr`, `Elf64_Ehdr`, `Elf32_Phdr`, `Elf64_Phdr`,
  `Elf32_Shdr`, `Elf64_Shdr`, `Elf32_Sym`, `Elf64_Sym`, `Elf32_Rel`,
  `Elf64_Rel`, `Elf32_Rela`, `Elf64_Rela`, `Elf32_Dyn`, `Elf64_Dyn`,
  `Elf_Nhdr`
- **Mach-O (15):** `MACH_HEADER`, `MACH_HEADER_BE`, `MACH_HEADER_64`,
  `MACH_HEADER_64_BE`, `FAT_HEADER`, `FAT_ARCH`, `LOAD_COMMAND`,
  `SEGMENT_COMMAND`, `SEGMENT_COMMAND_64`, `SECTION`, `SECTION_64`,
  `SYMTAB_COMMAND`, `DYLIB_COMMAND`, `DYLD_INFO_COMMAND`, `MAIN_COMMAND`
- **ZIP (8):** `ZIP_LOCAL_FILE_HEADER`, `ZIP_CENTRAL_DIRECTORY`,
  `ZIP_END_OF_CENTRAL_DIRECTORY`, `ZIP64_EOCD_RECORD`,
  `ZIP64_EOCD_LOCATOR`, `ZIP64_EXTRA_FIELD`, `ZIP_DATA_DESCRIPTOR`,
  `ZIP64_DATA_DESCRIPTOR`
- **Common (2):** `GUID`, `FILETIME`

The ELF templates use a built-in endianness-switch field that reads
`e_ident[EI_DATA]` to select little/big-endian parsing for the remainder
of the header. `MACH_HEADER_BE` / `MACH_HEADER_64_BE` handle the
byte-swapped magic variants used on PowerPC binaries.

In [ ]:
doc = HexDocument.open_bytes(b"\x00" * 16)
templates = doc.list_templates()
print(f"Built-in templates: {len(templates)}")
assert len(templates) == 48

expected = [
    "IMAGE_DOS_HEADER", "IMAGE_FILE_HEADER",
    "IMAGE_OPTIONAL_HEADER32", "IMAGE_OPTIONAL_HEADER64",
    "IMAGE_SECTION_HEADER", "IMAGE_DATA_DIRECTORY",
    "IMAGE_IMPORT_DESCRIPTOR", "IMAGE_EXPORT_DIRECTORY",
    "Elf32_Ehdr", "Elf64_Ehdr", "Elf32_Phdr", "Elf64_Phdr",
    "Elf32_Shdr", "Elf64_Shdr",
    "Elf32_Sym", "Elf64_Sym", "Elf32_Rel", "Elf64_Rel",
    "Elf32_Rela", "Elf64_Rela", "Elf32_Dyn", "Elf64_Dyn", "Elf_Nhdr",
    "MACH_HEADER", "MACH_HEADER_BE", "MACH_HEADER_64", "MACH_HEADER_64_BE",
    "FAT_HEADER", "FAT_ARCH",
    "LOAD_COMMAND", "SEGMENT_COMMAND", "SEGMENT_COMMAND_64",
    "SECTION", "SECTION_64",
    "SYMTAB_COMMAND", "DYLIB_COMMAND", "DYLD_INFO_COMMAND", "MAIN_COMMAND",
    "ZIP_LOCAL_FILE_HEADER", "ZIP_CENTRAL_DIRECTORY",
    "ZIP_END_OF_CENTRAL_DIRECTORY",
    "ZIP64_EOCD_RECORD", "ZIP64_EOCD_LOCATOR", "ZIP64_EXTRA_FIELD",
    "ZIP_DATA_DESCRIPTOR", "ZIP64_DATA_DESCRIPTOR",
    "GUID", "FILETIME",
]
template_names = {name for name, _ in templates}
missing = [t for t in expected if t not in template_names]
assert not missing, f"Missing templates: {missing}"
print(f"All {len(expected)} expected templates present")

detailed = doc.list_templates_detailed()
assert len(detailed) == len(templates)
categories = {cat for _, _, cat, _ in detailed}
assert {"ELF", "Mach-O"}.issubset(categories)
preview = [d for d in detailed if d[0] in {
    "IMAGE_DOS_HEADER", "Elf64_Ehdr", "MACH_HEADER_64",
    "ZIP_LOCAL_FILE_HEADER", "GUID",
}]
for name, desc, category, field_count in preview:
    print(f"  {name} [{category}]: {field_count} fields - {desc}")

In [ ]:
pe_data = bytearray(128)
pe_data[0:2] = b"MZ"
pe_data[60:64] = (0x80).to_bytes(4, "little")

doc = HexDocument.open_bytes(bytes(pe_data))
fields = doc.apply_template("IMAGE_DOS_HEADER", 0)
assert len(fields) > 0
assert fields[0]["name"] == "e_magic"
print(f"IMAGE_DOS_HEADER: {len(fields)} fields")
for f in fields[:5]:
    print(f"  {f['name']}: {f['display_value']} (offset={f['offset']}, size={f['size']})")

elf_data = bytearray(64)
elf_data[0:4] = b"\x7fELF"
elf_data[4] = 2
elf_data[5] = 1

doc2 = HexDocument.open_bytes(bytes(elf_data))
elf_fields = doc2.apply_template("Elf64_Ehdr", 0)
assert len(elf_fields) > 0
assert elf_fields[0]["name"] == "e_ident"
print(f"\nElf64_Ehdr: {len(elf_fields)} fields")
for f in elf_fields[:5]:
    print(f"  {f['name']}: {f['display_value']}")

zip_data = bytearray(30)
zip_data[0:4] = b"PK\x03\x04"
zip_data[4:6] = (20).to_bytes(2, "little")

doc3 = HexDocument.open_bytes(bytes(zip_data))
zip_fields = doc3.apply_template("ZIP_LOCAL_FILE_HEADER", 0)
assert len(zip_fields) > 0
print(f"\nZIP_LOCAL_FILE_HEADER: {len(zip_fields)} fields")
for f in zip_fields[:5]:
    print(f"  {f['name']}: {f['display_value']}")

macho_data = bytearray(32)
macho_data[0:4] = b"\xCF\xFA\xED\xFE"
macho_data[4:8] = (0x01000007).to_bytes(4, "little")

doc4 = HexDocument.open_bytes(bytes(macho_data))
macho_fields = doc4.apply_template("MACH_HEADER_64", 0)
assert len(macho_fields) > 0
print(f"\nMACH_HEADER_64: {len(macho_fields)} fields")
for f in macho_fields[:5]:
    print(f"  {f['name']}: {f['display_value']}")

zip64_data = bytearray(56)
zip64_data[0:4] = b"PK\x06\x06"
zip64_data[4:12] = (44).to_bytes(8, "little")

doc5 = HexDocument.open_bytes(bytes(zip64_data))
zip64_fields = doc5.apply_template("ZIP64_EOCD_RECORD", 0)
assert len(zip64_fields) > 0
print(f"\nZIP64_EOCD_RECORD: {len(zip64_fields)} fields")

print("\nTemplate application OK")

In [ ]:
import json

custom_template = {
    "name": "CUSTOM_HEADER",
    "description": "Custom binary header for tutorial",
    "default_endianness": "little",
    "fields": [
        {"name": "magic", "field_type": {"type": "UInt32"}, "description": "Magic number"},
        {"name": "version", "field_type": {"type": "UInt16"}, "description": "Version"},
        {"name": "flags", "field_type": {"type": "UInt16"}, "description": "Flags"},
        {"name": "name", "field_type": {"type": "FixedString", "params": 8}, "description": "Name"},
    ],
}

doc = HexDocument.open_bytes(b"\xDE\xAD\xBE\xEF\x01\x00\xFF\x00TestName")
name = doc.register_json_template(json.dumps(custom_template))
assert name == "CUSTOM_HEADER"

fields = doc.apply_template("CUSTOM_HEADER", 0)
assert len(fields) == 4
assert fields[0]["name"] == "magic"
print(f"Custom template '{name}': {len(fields)} fields")
for f in fields:
    print(f"  {f['name']}: {f['display_value']}")

exported_json = doc.export_template_json("CUSTOM_HEADER")
assert "CUSTOM_HEADER" in exported_json
print(f"\nExported JSON: {len(exported_json)} chars")

assert doc.remove_template("CUSTOM_HEADER")
assert not doc.remove_template("CUSTOM_HEADER")
print("Custom template register/export/remove OK")

## 17. Bookmarks

- `add_bookmark(offset, length, label, color)` → `int` (index)
- `list_bookmarks()` → `list[tuple[offset, length, label, color]]`
- `remove_bookmark(index)` → `bool`

The `Bookmark` class is also exposed directly for construction outside a
document; it is a plain value type with `offset`, `length`, `label`, `color`
attributes.

In [ ]:
doc = HexDocument.open_bytes(b"ABCDEFGHIJKLMNOP")

idx0 = doc.add_bookmark(0, 4, "Header", "#FF0000")
idx1 = doc.add_bookmark(4, 4, "Data", "#00FF00")
idx2 = doc.add_bookmark(8, 4, "Footer", "#0000FF")
assert idx0 == 0
assert idx1 == 1
assert idx2 == 2

bookmarks = doc.list_bookmarks()
assert len(bookmarks) == 3
assert bookmarks[0] == (0, 4, "Header", "#FF0000")
assert bookmarks[1] == (4, 4, "Data", "#00FF00")
assert bookmarks[2] == (8, 4, "Footer", "#0000FF")
print(f"Bookmarks: {bookmarks}")

assert doc.remove_bookmark(1)
bookmarks = doc.list_bookmarks()
assert len(bookmarks) == 2
print(f"After removing index 1: {bookmarks}")

assert not doc.remove_bookmark(99)

standalone = Bookmark(16, 8, "Footer data", "#808080")
assert standalone.offset == 16
assert standalone.length == 8
assert standalone.label == "Footer data"
assert standalone.color == "#808080"
print(f"Bookmark() repr: {standalone!r}")

print("Bookmarks OK")

## 18. Patch Export / Import

Hexcore derives patch records from the undo stack's overwrite entries. Six
export formats are provided, plus import for IPS/IPS32/BPS/UPS.

- `get_patches()` → `list[tuple[offset, bytes]]` — raw overwrite records
- `export_patches_ips()` → `bytes` — classic IPS (PATCH…EOF header/footer)
- `export_patches_ips32()` → `bytes` — 32-bit IPS (IPS32…EEOF)
- `export_patches_cod()` → `bytes` — compact binary COD format (per record:
  4-byte BE offset + 4-byte BE length + raw payload bytes, no header/footer)
- `export_patches_json()` → `str` — JSON list of `{offset, data: hex}` entries
- `import_patches_ips(data)` → `int` — accepts IPS or IPS32
- `export_patches_bps(source_data)` / `import_patches_bps(patch, source)`
- `export_patches_ups(source_data)` / `import_patches_ups(patch, source)`
- `export_patches_bps_from_path(source_path)` /
  `export_patches_ups_from_path(source_path)` — identical output to the
  in-memory exporters, but memory-map the source file instead of receiving
  it as a `bytes` object

IPS32 terminator-offset collisions (patches at `0x45454F46`, the
byte-encoded value of `EEOF`) are split into safe records rather than
producing a malformed file. BPS and UPS fail loud on out-of-bounds source
references instead of silently truncating.

In [ ]:
original = b"ABCDEFGHIJKLMNOP"
doc = HexDocument.open_bytes(original)
doc.write_bytes(0, b"XY")
doc.write_bytes(8, b"ZZ")

patches = doc.get_patches()
assert len(patches) >= 2
assert all(isinstance(offset, int) and isinstance(blob, bytes) for offset, blob in patches)
print(f"Raw patches: {patches}")

ips_data = doc.export_patches_ips()
assert isinstance(ips_data, bytes)
assert ips_data[:5] == b"PATCH"
assert ips_data[-3:] == b"EOF"
print(f"IPS export: {len(ips_data)} bytes")

ips32_data = doc.export_patches_ips32()
assert ips32_data[:5] == b"IPS32"
assert ips32_data[-4:] == b"EEOF"
print(f"IPS32 export: {len(ips32_data)} bytes")

cod_data = doc.export_patches_cod()
assert isinstance(cod_data, bytes)
assert len(cod_data) >= 8
first_offset = int.from_bytes(cod_data[0:4], "big")
first_length = int.from_bytes(cod_data[4:8], "big")
first_payload = cod_data[8:8 + first_length]
assert first_offset == patches[0][0]
assert first_payload == patches[0][1]
print(f"COD export: {len(cod_data)} bytes; "
      f"first record offset={first_offset}, length={first_length}, "
      f"payload={first_payload!r}")

json_data = doc.export_patches_json()
assert isinstance(json_data, str)
json_decoded = json.loads(json_data)
assert isinstance(json_decoded, list)
assert len(json_decoded) >= 2
assert "offset" in json_decoded[0]
assert "data" in json_decoded[0]
assert bytes.fromhex(json_decoded[0]["data"]) == patches[0][1]
print(f"JSON export: {len(json_data)} chars, {len(json_decoded)} records, "
      f"first record keys: {sorted(json_decoded[0].keys())}")

doc2 = HexDocument.open_bytes(original)
count = doc2.import_patches_ips(ips_data)
assert count >= 2
patched = doc2.read(0, 16)
assert patched[:2] == b"XY"
assert patched[8:10] == b"ZZ"
print(f"IPS import: {count} patches, result: {patched}")

doc3 = HexDocument.open_bytes(original)
count32 = doc3.import_patches_ips(ips32_data)
assert count32 >= 2
assert doc3.read(0, 2) == b"XY"
print(f"IPS32 import via import_patches_ips: {count32} patches")

bps_data = doc.export_patches_bps(original)
assert isinstance(bps_data, bytes)
assert bps_data[:4] == b"BPS1"
doc4 = HexDocument.open_bytes(b"")
target_len = doc4.import_patches_bps(bps_data, original)
assert target_len == len(original)
assert doc4.read(0, 16) == doc.read(0, 16)
print(f"BPS roundtrip: {len(bps_data)} bytes -> {target_len}-byte target")

ups_data = doc.export_patches_ups(original)
assert isinstance(ups_data, bytes)
assert ups_data[:4] == b"UPS1"
doc5 = HexDocument.open_bytes(b"")
target_len_ups = doc5.import_patches_ups(ups_data, original)
assert target_len_ups == len(original)
assert doc5.read(0, 16) == doc.read(0, 16)
print(f"UPS roundtrip: {len(ups_data)} bytes -> {target_len_ups}-byte target")

import os
import tempfile

with tempfile.NamedTemporaryFile(suffix=".bin", delete=False) as src_file:
    src_file.write(original)
    src_path = src_file.name
try:
    bps_from_path = doc.export_patches_bps_from_path(src_path)
    ups_from_path = doc.export_patches_ups_from_path(src_path)
    assert bps_from_path == bps_data
    assert ups_from_path == ups_data
    print(f"export_patches_bps_from_path matches in-memory export: {len(bps_from_path)} bytes")
    print(f"export_patches_ups_from_path matches in-memory export: {len(ups_from_path)} bytes")
finally:
    os.unlink(src_path)

print("Patch export/import OK")

## 19. Block Operations

Mutating helpers for moving and filling contiguous regions. All four
operations require every accessed region to fit inside the document and
record their changes on the undo stack.

- `fill_block(offset, length, pattern)` — `pattern` cycles to fill
  `length` bytes
- `copy_block(src_offset, length, dst_offset)`
- `move_block(src_offset, length, dst_offset)` — destination overwritten,
  source zeroed; overlapping source/destination ranges raise `ValueError`
- `swap_blocks(offset_a, len_a, offset_b, len_b)` — requires equal-length,
  non-overlapping blocks; unequal lengths or overlapping ranges raise
  `ValueError`

In [ ]:
doc = HexDocument.open_bytes(b"ABCDEFGHIJKLMNOP")
doc.fill_block(0, 4, b"\x55")
assert doc.read(0, 4) == b"\x55\x55\x55\x55"
assert doc.read(4, 4) == b"EFGH"
print(f"fill_block single byte: {doc.read(0, 16)}")

doc_pattern = HexDocument.open_bytes(b"\x00" * 10)
doc_pattern.fill_block(2, 6, b"AB")
assert doc_pattern.read(2, 6) == b"ABABAB"
print(f"fill_block pattern cycle: {doc_pattern.read(0, 10)}")

doc_copy = HexDocument.open_bytes(b"ABCDEFGHIJ")
doc_copy.copy_block(0, 3, 5)
assert doc_copy.read(5, 3) == b"ABC"
assert doc_copy.read(0, 3) == b"ABC"
print(f"copy_block: {doc_copy.read(0, 10)}")

doc_move = HexDocument.open_bytes(b"ABCDEFGHIJ")
doc_move.move_block(0, 3, 5)
assert doc_move.read(5, 3) == b"ABC"
assert doc_move.read(0, 3) == b"\x00\x00\x00"
print(f"move_block: {doc_move.read(0, 10)}")

doc_swap = HexDocument.open_bytes(b"ABCDEFGHIJ")
doc_swap.swap_blocks(0, 3, 5, 3)
assert doc_swap.read(0, 3) == b"FGH"
assert doc_swap.read(5, 3) == b"ABC"
print(f"swap_blocks equal size: {doc_swap.read(0, 10)}")

doc_swap_uneven = HexDocument.open_bytes(b"ABCDEFGHIJ")
try:
    doc_swap_uneven.swap_blocks(0, 2, 5, 4)
except ValueError as exc:
    print(f"swap_blocks unequal lengths rejected: {type(exc).__name__}")
else:
    raise AssertionError("swap_blocks must reject unequal-length blocks")

doc_overlap = HexDocument.open_bytes(b"ABCDEF")
try:
    doc_overlap.swap_blocks(0, 4, 2, 2)
except ValueError as exc:
    print(f"swap_blocks overlapping rejected: {type(exc).__name__}")
else:
    raise AssertionError("swap_blocks must reject overlapping ranges")

assert doc_copy.undo()
assert doc_copy.read(0, 10) == b"ABCDEFGHIJ"
print("copy_block undo restores original bytes")

print("Block operations OK")

## 20. Bit Operations

Individual-bit read/write/toggle inside a byte. `bit_index` must be in
`0..=7`; LSB is index 0.

- `get_bit(offset, bit_index)` → `bool`
- `set_bit(offset, bit_index, value)` — records undo
- `toggle_bit(offset, bit_index)` → `bool` (new value)

In [ ]:
doc = HexDocument.open_bytes(bytes([0b10101010, 0x00, 0xFF]))

assert doc.get_bit(0, 0) is False
assert doc.get_bit(0, 1) is True
assert doc.get_bit(0, 7) is True
print(f"get_bit sweep (byte 0): {[doc.get_bit(0, i) for i in range(8)]}")

doc.set_bit(1, 3, True)
assert doc.read_byte(1) == 0b00001000
doc.set_bit(1, 3, False)
assert doc.read_byte(1) == 0x00
print("set_bit on/off OK")

new_val = doc.toggle_bit(2, 0)
assert new_val is False
assert doc.read_byte(2) == 0xFE
new_val = doc.toggle_bit(2, 0)
assert new_val is True
assert doc.read_byte(2) == 0xFF
print("toggle_bit returns new value OK")

for bad in (8, 99, 255):
    try:
        doc.get_bit(0, bad)
    except ValueError as exc:
        print(f"get_bit(0, {bad}) rejected: {type(exc).__name__}")
    else:
        raise AssertionError(f"get_bit(0, {bad}) must be rejected")

print("Bit operations OK")

## 21. Virtual Address Mapping

Associate ranges of the file with runtime virtual addresses so analysis
layers can translate between the two spaces.

- `add_va_mapping(file_offset, virtual_address, length)` — mappings are
  kept sorted by `file_offset`
- `remove_va_mapping(index)` → `bool`
- `list_va_mappings()` → `list[tuple[file_offset, virtual_address, length]]`
- `file_offset_to_va(offset)` → `int | None`
- `va_to_file_offset(va)` → `int | None`

In [ ]:
doc = HexDocument.open_bytes(b"\x00" * 0x2000)
doc.add_va_mapping(0x0000, 0x00400000, 0x1000)
doc.add_va_mapping(0x1000, 0x00500000, 0x1000)

mappings = doc.list_va_mappings()
assert mappings == [(0x0000, 0x00400000, 0x1000), (0x1000, 0x00500000, 0x1000)]
print(f"VA mappings: {mappings}")

assert doc.file_offset_to_va(0x0000) == 0x00400000
assert doc.file_offset_to_va(0x0FFF) == 0x00400FFF
assert doc.file_offset_to_va(0x1000) == 0x00500000
assert doc.file_offset_to_va(0x1FFF) == 0x00500FFF
assert doc.file_offset_to_va(0x2000) is None
print("file_offset_to_va OK")

assert doc.va_to_file_offset(0x00400100) == 0x0100
assert doc.va_to_file_offset(0x00500100) == 0x1100
assert doc.va_to_file_offset(0x00700000) is None
print("va_to_file_offset OK")

assert doc.remove_va_mapping(0)
assert len(doc.list_va_mappings()) == 1
assert not doc.remove_va_mapping(99)

print("Virtual address mapping OK")

## 22. String Extraction

`extract_strings(min_length, include_ascii, include_utf16, max_results)`
returns a `list[dict]` where each entry has:
- `offset: int`
- `length: int` (bytes, not characters, for UTF-16)
- `encoding: str` (`"ascii"` or `"utf16le"`)
- `content: str`

ASCII printable characters include tab/CR/LF. The UTF-16LE extractor
supports the full Unicode range, including surrogate pairs and characters
beyond the Basic Multilingual Plane.

In [ ]:
ascii_payload = (
    b"\x00\x01Hello\x00World\x00\x02"
    b"short\x00\x00A long string exceeding the minimum length\x00tail"
)
doc = HexDocument.open_bytes(ascii_payload)
ascii_strings = doc.extract_strings(5, True, False, 16)
assert isinstance(ascii_strings, list)
assert all(s["encoding"] == "ascii" for s in ascii_strings)
contents = [s["content"] for s in ascii_strings]
assert "Hello" in contents
assert "World" in contents
assert "short" in contents
assert any("exceeding" in c for c in contents)
print(f"ASCII strings found: {len(ascii_strings)}")
for s in ascii_strings:
    print(f"  @{s['offset']:#06x} [{s['encoding']}, {s['length']}B] {s['content']!r}")

utf16_payload = (
    "Hello\u0000".encode("utf-16-le") +
    b"\x00" * 8 +
    "\u30d5\u30a1\u30a4\u30eb\u0000".encode("utf-16-le") +
    b"\x00" * 4 +
    "\U0001F600 emoji".encode("utf-16-le")
)
doc2 = HexDocument.open_bytes(utf16_payload)
utf16_strings = doc2.extract_strings(3, False, True, 16)
assert all(s["encoding"] == "utf16le" for s in utf16_strings)
utf16_contents = {s["content"] for s in utf16_strings}
assert "Hello" in utf16_contents
assert "\u30d5\u30a1\u30a4\u30eb" in utf16_contents
assert any("\U0001F600" in c for c in utf16_contents)
print(f"\nUTF-16LE strings found: {len(utf16_strings)}")
for s in utf16_strings:
    print(f"  @{s['offset']:#06x} [{s['encoding']}, {s['length']}B] {s['content']!r}")

mixed_strings = doc2.extract_strings(3, True, True, 16)
mixed_encodings = {s["encoding"] for s in mixed_strings}
assert "utf16le" in mixed_encodings
print(f"\nCombined scan encodings: {sorted(mixed_encodings)}")

print("String extraction OK")

## 23. PE Checksum Verify / Repair

- `verify_pe_checksum()` → `dict` with `stored`, `calculated`, `offset`,
  `valid`. Raises `ValueError` if the document is not a valid PE file.
- `repair_pe_checksum()` — writes the calculated value into the PE
  `OptionalHeader.CheckSum` field, recording an undo entry.

In [ ]:
pe = bytearray(0x200)
pe[0:2] = b"MZ"
pe_header_offset = 0x80
pe[0x3C:0x40] = pe_header_offset.to_bytes(4, "little")
pe[pe_header_offset:pe_header_offset + 4] = b"PE\x00\x00"
pe[pe_header_offset + 4:pe_header_offset + 6] = (0x8664).to_bytes(2, "little")
pe[pe_header_offset + 0x18:pe_header_offset + 0x1A] = (0x020B).to_bytes(2, "little")
checksum_offset = pe_header_offset + 0x58
pe[checksum_offset:checksum_offset + 4] = (0xDEADBEEF).to_bytes(4, "little")
for i in range(pe_header_offset + 0x100, 0x200):
    pe[i] = (i * 17) & 0xFF

doc = HexDocument.open_bytes(bytes(pe))
result = doc.verify_pe_checksum()
assert isinstance(result, dict)
assert result["offset"] == checksum_offset
assert result["stored"] == 0xDEADBEEF
assert result["valid"] is False
print(f"verify_pe_checksum: stored=0x{result['stored']:08X}, "
      f"calculated=0x{result['calculated']:08X}, offset={result['offset']}")

doc.repair_pe_checksum()
repaired = doc.verify_pe_checksum()
assert repaired["valid"] is True
assert repaired["stored"] == repaired["calculated"]
print(f"repair_pe_checksum: now valid = {repaired['valid']}")

assert doc.undo()
after_undo = doc.verify_pe_checksum()
assert after_undo["stored"] == 0xDEADBEEF
print("repair_pe_checksum undo restores original stored value")

doc_bad = HexDocument.open_bytes(b"not a pe file")
try:
    doc_bad.verify_pe_checksum()
except ValueError as exc:
    print(f"verify_pe_checksum on non-PE rejected: {type(exc).__name__}")
else:
    raise AssertionError("verify_pe_checksum must reject non-PE input")

print("PE checksum OK")

## 24. Process Memory (Windows only)

On Windows, a running process's address space can be attached to and read
as though it were a file. Non-Windows platforms raise `RuntimeError`.

- `HexDocument.from_process_memory(pid, address, size)` — reads `size`
  bytes starting at `address` in the target process into a new document
- `HexDocument.list_process_memory_regions(pid)` →
  `list[tuple[base_address, size, protection, state]]`

Requires the current process to have sufficient privileges to call
`OpenProcess` with `PROCESS_VM_READ` / `PROCESS_QUERY_INFORMATION` on the
target.

In [ ]:
import os
import platform

if platform.system() == "Windows":
    pid = os.getpid()
    regions = HexDocument.list_process_memory_regions(pid)
    assert isinstance(regions, list)
    assert len(regions) > 0
    assert all(len(r) == 4 for r in regions)
    base_address, size, protection, state = regions[0]
    assert isinstance(base_address, int)
    assert isinstance(size, int)
    assert isinstance(protection, int)
    assert isinstance(state, int)
    print(f"Process memory regions for pid {pid}: {len(regions)}")
    print(f"  first region: base=0x{base_address:X}, size={size}, "
          f"protection=0x{protection:X}, state=0x{state:X}")

    readable = next(
        (r for r in regions if r[1] >= 4096 and r[3] == 0x1000),
        None,
    )
    if readable is not None:
        base, _, _, _ = readable
        mem_doc = HexDocument.from_process_memory(pid, base, 4096)
        assert mem_doc.length() == 4096
        print(f"  from_process_memory: read 4096 bytes from 0x{base:X}")
    else:
        print("  no committed region of sufficient size found for read demo")
else:
    try:
        HexDocument.list_process_memory_regions(0)
    except RuntimeError as exc:
        print(f"Non-Windows platform correctly reports: {exc}")

print("Process memory OK")

## 25. Large File Memory Controls

Hints that the UI / analysis layer can read to budget for large files.
They do not force a specific chunking strategy on the internal piece
table but are honored by streaming consumers.

- `get_document_memory_usage()` → `int` — current document size in bytes
- `get_chunk_size_hint()` / `set_chunk_size_hint(size)` — default 4 MiB
- `get_memory_budget_hint()` / `set_memory_budget_hint(budget)` — default
  512 MiB

In [ ]:
doc = HexDocument.open_bytes(b"A" * 4096)
assert doc.get_document_memory_usage() == 4096

assert doc.get_chunk_size_hint() == 4 * 1024 * 1024
assert doc.get_memory_budget_hint() == 512 * 1024 * 1024
print(f"Defaults: chunk={doc.get_chunk_size_hint()}, "
      f"budget={doc.get_memory_budget_hint()}")

doc.set_chunk_size_hint(64 * 1024)
doc.set_memory_budget_hint(128 * 1024 * 1024)
assert doc.get_chunk_size_hint() == 64 * 1024
assert doc.get_memory_budget_hint() == 128 * 1024 * 1024
print(f"After override: chunk={doc.get_chunk_size_hint()}, "
      f"budget={doc.get_memory_budget_hint()}")

doc.insert_bytes(doc.length(), b"B" * 1024)
assert doc.get_document_memory_usage() == 4096 + 1024
print(f"document_memory_usage after insert: {doc.get_document_memory_usage()}")

print("Large file controls OK")

## 26. Binary Diff

Module-level functions backed by a Myers-style edit script (`similar` crate)
for payloads ≤ 1 MiB and a block-anchored algorithm for larger inputs.

- `diff_bytes(data_a, data_b)` → `dict`
- `diff_files(path_a, path_b)` → same

The returned dict contains:
- `total_differences: int`
- `files_identical: bool`
- `regions: list[dict]` with `offset_a`, `offset_b`, `length`, and
  `diff_type` ∈ {`match`, `modified`, `inserted_a`, `inserted_b`}

In [ ]:
import os
import tempfile

result = diff_bytes(HELLO_WORLD, b"Hello Brave World!")
assert not result["files_identical"]
assert result["total_differences"] > 0
diff_types = {r["diff_type"] for r in result["regions"]}
assert "match" in diff_types
assert diff_types & {"inserted_a", "inserted_b", "modified"}
print("diff_bytes:")
print(f"  identical: {result['files_identical']}")
print(f"  total_differences: {result['total_differences']}")
for r in result["regions"]:
    print(f"  type={r['diff_type']}, offset_a={r['offset_a']}, "
          f"offset_b={r['offset_b']}, len={r['length']}")

result2 = diff_bytes(b"same", b"same")
assert result2["files_identical"]
assert result2["total_differences"] == 0
print(f"\nIdentical: files_identical={result2['files_identical']}")

with tempfile.NamedTemporaryFile(suffix=".bin", delete=False) as f1:
    f1.write(b"File content A")
    path1 = f1.name
with tempfile.NamedTemporaryFile(suffix=".bin", delete=False) as f2:
    f2.write(b"File content B")
    path2 = f2.name
try:
    file_result = diff_files(path1, path2)
    assert not file_result["files_identical"]
    print(f"\ndiff_files: identical={file_result['files_identical']}, "
          f"diffs={file_result['total_differences']}")
finally:
    os.unlink(path1)
    os.unlink(path2)

print("Binary diff OK")

## 27. Byte Statistics

`byte_statistics()` → `list[tuple[byte_value, count]]` (256 entries, ordered
by byte value).

In [ ]:
doc = HexDocument.open_bytes(b"AAABBC")
stats = doc.byte_statistics()
assert len(stats) == 256

stats_dict = dict(stats)
assert stats_dict[0x41] == 3
assert stats_dict[0x42] == 2
assert stats_dict[0x43] == 1
assert stats_dict[0x00] == 0

non_zero = [(bv, c) for bv, c in stats if c > 0]
print("Byte statistics for b'AAABBC':")
for bv, c in non_zero:
    print(f"  0x{bv:02X} ('{chr(bv)}'): {c}")

total = sum(c for _, c in stats)
assert total == 6
print(f"Total bytes: {total}")
print("Byte statistics OK")

## 28. Error Handling

Hexcore raises Python exceptions for invalid operations. This section
verifies that the expected error paths produce the correct exception
types.

- Out-of-range read/write/insert/delete → `ValueError`
- Unknown hash/transform/template/encoding → `ValueError`
- AES with bad key size → `ValueError`
- Missing required transform parameter → `ValueError`
- Custom CRC with invalid width → `ValueError`
- Invalid hash range → `ValueError`
- Bit index > 7 → `ValueError`
- Block operations out of bounds / overlapping swaps → `ValueError`
- `verify_pe_checksum` on non-PE data → `ValueError`
- `open()` on a missing file → `OSError` (PyIOError)

In [ ]:
def expect_error(fn, description, expected=Exception):
    try:
        fn()
    except expected as exc:
        print(f"  {description}: {type(exc).__name__}")
        return True
    else:
        raise AssertionError(f"{description} did not raise")

doc = HexDocument.open_bytes(b"ABCD")

expect_error(lambda: doc.read(1000, 1), "read beyond end", ValueError)
expect_error(lambda: doc.write_bytes(1000, b"X"), "write beyond end", ValueError)
expect_error(lambda: doc.insert_bytes(1000, b"X"), "insert beyond end", ValueError)
expect_error(lambda: doc.delete_bytes(1000, 1), "delete beyond end", ValueError)
expect_error(lambda: doc.compute_hash("nonexistent"), "invalid hash algorithm", ValueError)
expect_error(
    lambda: doc.transform_data("nonexistent", 0, 4, {}),
    "invalid transform", ValueError,
)
expect_error(lambda: doc.apply_template("NONEXISTENT", 0), "invalid template", ValueError)
expect_error(
    lambda: doc.decode_text(0, 4, "nonexistent_enc"),
    "invalid encoding", ValueError,
)
expect_error(
    lambda: doc.transform_data("aes_ecb_encrypt", 0, 4, {"key": b"\x00" * 7, "padding": b"none"}),
    "AES with invalid key size", ValueError,
)
expect_error(
    lambda: doc.transform_data("aes_ecb_encrypt", 0, 4, {"key": b"\x00" * 16, "padding": b"bogus"}),
    "AES with invalid padding mode", ValueError,
)
expect_error(
    lambda: doc.transform_data("xor_single", 0, 4, {}),
    "missing xor key", ValueError,
)
expect_error(
    lambda: doc.transform_data("bit_shift_left", 0, 4, {"count": b"\x08"}),
    "bit shift count > 7", ValueError,
)
expect_error(
    lambda: doc.compute_hash_custom_crc((0, 4), 0x04C11DB7, 0, 12, (False, False), 0),
    "invalid CRC width", ValueError,
)
expect_error(
    lambda: doc.compute_hash_range(10, 5, "md5"),
    "invalid hash range", ValueError,
)
expect_error(lambda: doc.get_bit(0, 8), "bit_index > 7", ValueError)
expect_error(lambda: doc.set_bit(0, 9, True), "set_bit index > 7", ValueError)
expect_error(
    lambda: doc.fill_block(0, 100, b"\x00"),
    "fill_block past end", ValueError,
)
expect_error(
    lambda: doc.fill_block(0, 2, b""),
    "fill_block empty pattern", ValueError,
)
expect_error(
    lambda: doc.swap_blocks(0, 3, 1, 3),
    "swap_blocks overlapping", ValueError,
)
expect_error(
    lambda: HexDocument.open_bytes(b"not pe").verify_pe_checksum(),
    "verify_pe_checksum on non-PE", ValueError,
)
expect_error(
    lambda: HexDocument.open("/nonexistent/path/file.bin"),
    "open nonexistent file", OSError,
)

print("\nAll error cases handled correctly")

## Summary

In [ ]:
print("=" * 60)
print("HEXCORE TUTORIAL COMPLETE")
print("=" * 60)
print("All sections passed:")
print("  Document I/O (new, open, open_bytes, save, save_as)")
print("  Read / Write / Insert / Delete")
print("  Undo / Redo")
print("  Byte & hex search (with per-nibble wildcards)")
print("  Text & encoded-text search (case-sensitive and insensitive)")
print("  Regex search")
print("  Numeric search (int, float, range)")
print("  Find & replace")
print("  Data inspector (40+ interpretation keys)")
print("  24 hash algorithms with test vectors")
print("  Hash range & custom CRC")
print("  Entropy analysis (6 functions)")
print("  23 transforms across 7 categories (4 AES padding modes)")
print("  36 encodings (strict ASCII) with roundtrip verification")
print("  48 built-in templates (PE / ELF / Mach-O / ZIP / ZIP64 / common)")
print("  Custom JSON templates")
print("  Bookmarks")
print("  IPS / IPS32 / COD / JSON / BPS / UPS patch export & import")
print("  Block operations (fill / copy / move / swap)")
print("  Bit operations (get / set / toggle)")
print("  Virtual address mapping (file_offset <-> virtual_address)")
print("  String extraction (ASCII + full-Unicode UTF-16LE)")
print("  PE checksum verify / repair")
print("  Process memory attach (Windows)")
print("  Large file memory controls")
print("  Binary diff (Myers-style bytes & files)")
print("  Byte statistics")
print("  Error handling")